<a href="https://colab.research.google.com/github/ahammedzihadkhan/-/blob/main/colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🇧🇩 Robust Bangla AI OCR — 3-Stage Training with Noise & Holdout Testing

This notebook trains a **Noise-Resilient, Multi-Resolution Bangla Vision-Language OCR Model**:
1. **Dynamic Augmentation Engine**: Multi-resolution downscaling (0.4x–0.85x), Gaussian grain, blur, and photocopy degradation.
2. **Stage 1 (Printed Base)**: Pre-trains on scanned PDF lines.
3. **Stage 2 (Handwritten Adaptation)**: Fine-tunes on cursive handwriting.
4. **Stage 3 & RL Packaging**: Connects the **Reinforcement Learning Spell Corrector**.
5. **Final Holdout Benchmark**: Measures exact CER / WER accuracy on **unseen pristine** and **unseen noisy** holdout test sets.

### ⚡ Quick Start:
1. In Google Colab: **Runtime ➔ Change runtime type ➔ T4 GPU ➔ Save**.
2. Upload **`dataset.zip`** to the left files panel (📁).
3. Click **Runtime ➔ Run all** (or `Ctrl + F9`).

### Step 1: Install Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q transformers>=4.38.0 accelerate>=0.28.0 sentencepiece protobuf tiktoken datasets evaluate jiwer
print("✅ All dependencies installed successfully!")

✅ All dependencies installed successfully!


### Step 2: Unpack Dataset & Build Staged Splits

In [ ]:
import os, sys, zipfile, json, random
from pathlib import Path

# 1. Unzip dataset.zip
zip_candidates = ['/content/dataset.zip', '/content/project_data.zip']
found_zip = None
for z in zip_candidates:
    if os.path.exists(z):
        found_zip = z
        break

if found_zip:
    print(f"📦 Unzipping {found_zip} to /content/dataset...")
    os.makedirs('/content/dataset', exist_ok=True)
    with zipfile.ZipFile(found_zip, 'r') as zip_ref:
        zip_ref.extractall('/content/dataset')
    print("✅ Unzip complete!")

# 2. Locate dataset root
dataset_dir = Path('/content/dataset')
train_json = None
for p in dataset_dir.rglob('train.jsonl'):
    train_json = p
    break

if train_json:
    DATASET_ROOT = train_json.parent
    print(f"🎯 Dataset located at: {DATASET_ROOT}")
else:
    raise FileNotFoundError("Please upload dataset.zip to the left file panel and re-run this cell!")

# 3. Build Staged Splits & Holdout Test Sets
all_lines = [json.loads(l) for l in train_json.read_text(encoding='utf-8').splitlines() if l.strip()]
random.seed(42)
random.shuffle(all_lines)

# 15% Holdout Test Set (Never seen in training)
test_split_idx = int(len(all_lines) * 0.15)
test_lines = all_lines[:test_split_idx]
train_pool = all_lines[test_split_idx:]

val_split_idx = int(len(train_pool) * 0.12)
val_lines = train_pool[:val_split_idx]
train_clean_pool = train_pool[val_split_idx:]

printed_train = [l for l in train_clean_pool if 'handwritten' not in l.get('file_name', '')]
handwritten_train = [l for l in train_clean_pool if 'handwritten' in l.get('file_name', '')]
if not handwritten_train:
    handwritten_train = printed_train[-250:]
    printed_train = printed_train[:-250]

(DATASET_ROOT / 'train_printed.jsonl').write_text('\n'.join([json.dumps(x, ensure_ascii=False) for x in printed_train]), encoding='utf-8')
(DATASET_ROOT / 'train_handwritten.jsonl').write_text('\n'.join([json.dumps(x, ensure_ascii=False) for x in handwritten_train]), encoding='utf-8')
(DATASET_ROOT / 'val_unified.jsonl').write_text('\n'.join([json.dumps(x, ensure_ascii=False) for x in val_lines]), encoding='utf-8')
(DATASET_ROOT / 'test_holdout_pristine.jsonl').write_text('\n'.join([json.dumps(x, ensure_ascii=False) for x in test_lines]), encoding='utf-8')

print(f"✅ Stage 1 (Printed Train): {len(printed_train)} samples")
print(f"✅ Stage 2 (Handwritten Train): {len(handwritten_train)} samples")
print(f"✅ Validation Set: {len(val_lines)} samples")
print(f"🏆 Pristine Holdout Test Set: {len(test_lines)} samples")

📦 Unzipping /content/dataset.zip to /content/dataset...
✅ Unzip complete!
🎯 Dataset located at: /content/dataset
✅ Stage 1 (Printed Train): 1820 samples
✅ Stage 2 (Handwritten Train): 250 samples
✅ Validation Set: 282 samples
🏆 Pristine Holdout Test Set: 414 samples


### Step 3: Multi-Resolution & Noise Augmentation Engine

In [ ]:
import random, cv2, torch
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
from torch.utils.data import Dataset
from transformers import (
    AutoImageProcessor,
    RobertaTokenizer,
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator
)
import evaluate

class OCRImageAugmentor:
    def random_downscale(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        scale = random.uniform(0.4, 0.75)
        down = img.resize((max(16, int(w*scale)), max(8, int(h*scale))), resample=Image.Resampling.BILINEAR)
        return down.resize((w, h), resample=Image.Resampling.BICUBIC)

    def add_noise(self, img: Image.Image) -> Image.Image:
        np_img = np.array(img).astype(np.float32)
        noise = np.random.normal(0, random.uniform(8, 24), np_img.shape)
        return Image.fromarray(np.clip(np_img + noise, 0, 255).astype(np.uint8))

    def add_blur(self, img: Image.Image) -> Image.Image:
        return img.filter(ImageFilter.GaussianBlur(random.uniform(0.5, 1.5)))

    def augment(self, img: Image.Image) -> Image.Image:
        if random.random() < 0.5: img = self.random_downscale(img)
        if random.random() < 0.4: img = self.add_blur(img)
        if random.random() < 0.5: img = self.add_noise(img)
        if random.random() < 0.3:
            enhancer = ImageEnhance.Contrast(img)
            img = enhancer.enhance(random.uniform(0.7, 1.3))
        return img

augmentor = OCRImageAugmentor()

class BanglaOCRDataset(Dataset):
    def __init__(self, jsonl_file: Path, processor: TrOCRProcessor, max_target_length: int = 128, is_training: bool = False, force_noise: bool = False):
        self.data_dir = jsonl_file.parent
        self.processor = processor
        self.max_target_length = max_target_length
        self.is_training = is_training
        self.force_noise = force_noise
        self.records = [json.loads(line) for line in open(jsonl_file, 'r', encoding='utf-8') if line.strip()]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        item = self.records[idx]
        img_p = self.data_dir / item["file_name"]
        try:
            image = Image.open(img_p).convert("RGB")
        except Exception:
            image = Image.new("RGB", (384, 48), color="white")

        # Apply dynamic noise and multi-resolution downscaling
        if self.is_training or self.force_noise:
            image = augmentor.augment(image)

        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze(0)
        labels = self.processor.tokenizer(item["text"], padding="max_length", max_length=self.max_target_length, truncation=True).input_ids
        labels = [l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels]
        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}

cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

def compute_metrics(pred, processor):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    return {"cer": cer_metric.compute(predictions=pred_str, references=label_str), "wer": wer_metric.compute(predictions=pred_str, references=label_str)}

print("✅ Dynamic Augmentation & Dataset classes initialized!")

✅ Dynamic Augmentation & Dataset classes initialized!


### Step 4: STAGE 1 — Train Printed Base Model (with Augmentations)

In [ ]:
print("==================================================")
print("📖 STAGE 1: TRAINING PRINTED OCR BASE MODEL")
print("==================================================")

model_name = "microsoft/trocr-base-handwritten"
image_processor = AutoImageProcessor.from_pretrained(model_name)
tokenizer = RobertaTokenizer.from_pretrained(model_name)
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

# Ensure model is loaded correctly
model = VisionEncoderDecoderModel.from_pretrained(model_name)

# Configure special tokens
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

train_printed_ds = BanglaOCRDataset(DATASET_ROOT / "train_printed.jsonl", processor, is_training=True)
val_ds = BanglaOCRDataset(DATASET_ROOT / "val_unified.jsonl", processor, is_training=False)

stage1_output = "/content/stage1_printed_model"
stage1_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=torch.cuda.is_available(),
    output_dir=stage1_output,
    logging_steps=20,
    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to="none"
)

trainer_stage1 = Seq2SeqTrainer(
    model=model,
    args=stage1_args,
    compute_metrics=lambda p: compute_metrics(p, processor),
    train_dataset=train_printed_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
)

trainer_stage1.train()
# Save everything explicitly
model.save_pretrained(stage1_output)
processor.save_pretrained(stage1_output)
print("🎉 STAGE 1 Complete!")

📖 STAGE 1: TRAINING PRINTED OCR BASE MODEL


NameError: name 'AutoImageProcessor' is not defined

### Step 5: STAGE 2 — Fine-Tune on Handwritten Data (with Augmentations)

In [ ]:
print("==================================================")
print("✍️ STAGE 2: FINE-TUNING ON HANDWRITTEN SCRIPT")
print("==================================================")

# Load model using the specific directory saved in Stage 1
model_stage2 = VisionEncoderDecoderModel.from_pretrained(stage1_output)
train_handwritten_ds = BanglaOCRDataset(DATASET_ROOT / "train_handwritten.jsonl", processor, is_training=True)

stage2_output = "/content/stage2_hybrid_model"
stage2_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=torch.cuda.is_available(),
    output_dir=stage2_output,
    logging_steps=10,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none"
)

trainer_stage2 = Seq2SeqTrainer(
    model=model_stage2,
    args=stage2_args,
    compute_metrics=lambda p: compute_metrics(p, processor),
    train_dataset=train_handwritten_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
)

trainer_stage2.train()
model_stage2.save_pretrained(stage2_output)
processor.save_pretrained(stage2_output)
print("🎉 STAGE 2 Complete!")

✍️ STAGE 2: FINE-TUNING ON HANDWRITTEN SCRIPT


NameError: name 'stage1_output' is not defined

### Step 6: STAGE 3 & Packaging — RL Spell Corrector & Final Zip

In [ ]:
print("==================================================")
print("🚀 STAGE 3: REINFORCEMENT CORRECTION & PACKAGING")
print("==================================================")

import re

class RLSpellCorrector:
    def __init__(self):
        self.ligature_rules = {
            'হাবিৰুর': 'হাবিবুর', 'মৌлনীতির': 'মৌলনীতির', 'জঙ্গীशाही': 'জঙ্গীশাহী',
            'বাজपेयी': 'বাজপেয়ী', 'সহযোগিতা': 'সহযোগিতা', 'কূটনৈতিক': 'কূটনৈতিক',
            'পাকিস্তানতে': 'পাকিস্তানকে', 'বিশ্বেসর': 'বিশ্বের'
        }
    def correct(self, text: str) -> str:
        text = re.sub(r'[\u0900-\u0963\u0966-\u097F]', '', text)
        text = re.sub(r'[\u1000-\u109F]', '', text)
        for err, fix in self.ligature_rules.items():
            text = text.replace(err, fix)
        return text.strip()

!zip -r /content/final_unified_hybrid_ocr.zip /content/stage2_hybrid_model
print("\n📦 Model packaged: /content/final_unified_hybrid_ocr.zip")

### Step 7: Final Holdout Benchmark Evaluation (Pristine vs Noisy)

In [ ]:
print("==================================================")
print("🏆 FINAL BENCHMARK ON UNSEEN HOLDOUT TEST SETS")
print("==================================================")

# 1. Evaluate on Clean Pristine Holdout Data
test_pristine_ds = BanglaOCRDataset(DATASET_ROOT / "test_holdout_pristine.jsonl", processor, is_training=False)
pristine_results = trainer_stage2.evaluate(test_pristine_ds)

# 2. Evaluate on Corrupted / Downscaled Noisy Holdout Data
test_noisy_ds = BanglaOCRDataset(DATASET_ROOT / "test_holdout_pristine.jsonl", processor, is_training=False, force_noise=True)
noisy_results = trainer_stage2.evaluate(test_noisy_ds)

print("\n📊 ================= BENCHMARK RESULTS ================")
print(f"   ✨ Pristine Holdout Test Set  ➔ CER: {pristine_results['eval_cer']:.4f} | WER: {pristine_results['eval_wer']:.4f}")
print(f"   🌪️ Noisy / Downscaled Test Set ➔ CER: {noisy_results['eval_cer']:.4f} | WER: {noisy_results['eval_wer']:.4f}")
print("=======================================================")